# Four-model pilot: DeepSeek, Qwen, Mistral, GPT-OSS

Run all performs setup, verifies a non-resetting OpenRouter key limit of at most $100, then runs
canary → full-context probe → launch checks → separate competence checks → four-market pilot for each model.
Two families run concurrently; a failure stops that family's sequence, while other families finish.
There is no automatic expansion. Up to 11,688 successful calls including checks, plus retries.

Upload `collusion-models.bundle` beside this notebook in Drive's `collusion-sprint` folder.
Put the key in Colab Secrets as `OPENROUTER_API_KEY`, with notebook access enabled.
Before starting, set this key's OpenRouter limit to at most $100 with no reset interval.
This notebook reads the limit; it does not change billing settings. Modal credits do not pay these calls.

The original `run.ipynb` remains usable for an existing DeepSeek run. Do not launch DeepSeek again here
if its pilot is already running: remove `deepseek31` from FAMILIES. Never change the code under a running game.
New journals use `runs/four_models_v1`. After a disconnect, use the same notebook, commit and directory.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount('/content/drive')
SOURCE = '/content/drive/MyDrive/collusion-sprint/collusion-models.bundle'
COMMIT = '062037d762eaab3157657210b2a2c41bad7f8d81'
REPO = Path('/content/collusion-models')
OUT = Path('/content/drive/MyDrive/collusion-sprint/runs/four_models_v1')
FAMILIES = ['deepseek31', 'qwen38', 'mistral3', 'gptoss120']

def sh(args, **kwargs):
    subprocess.run(args, check=True, **kwargs)

if not REPO.exists():
    sh(['git', 'clone', '-q', SOURCE, str(REPO)])
else:
    sh(['git', '-C', str(REPO), 'fetch', SOURCE])
sh(['git', '-C', str(REPO), 'checkout', '-q', COMMIT])
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sh(['bash', 'scripts/get_starter.sh'])
sh([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '-e', 'starter/util',
    '-e', 'starter/algorithmic-pricing-llms'])
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
OUT.mkdir(parents=True, exist_ok=True)
print('Pinned code:', COMMIT)


In [ ]:
import urllib.request


# Metadata request only: no model call, key never printed or saved.
def check_key_budget(data):
    limit, remaining = data.get('limit'), data.get('limit_remaining')
    if not isinstance(limit, (int, float)) or not 0 < limit <= 100 or data.get('limit_reset') is not None:
        raise RuntimeError('Set this key to a non-resetting limit of at most $100 in OpenRouter, then rerun.')
    if not isinstance(remaining, (int, float)) or remaining <= 0:
        raise RuntimeError('No remaining key allowance. Check spending; do not raise the $100 ceiling.')
    return {'key_limit_usd': limit, 'remaining_allowance_usd': remaining}

req = urllib.request.Request('https://openrouter.ai/api/v1/key',
    headers={'Authorization': 'Bearer ' + os.environ['OPENROUTER_API_KEY']})
with urllib.request.urlopen(req, timeout=30) as response:
    metadata = json.load(response)['data']
print(check_key_budget(metadata))
# Key allowance is a spending cap, not proof of sufficient prepaid account credit.
del metadata, req


In [ ]:
from datetime import UTC, datetime

from sprint.run import load_config


def endpoint_snapshot(config):
    model = config['model']
    url = 'https://openrouter.ai/api/v1/models/' + model.model + '/endpoints'
    with urllib.request.urlopen(url, timeout=30) as response:
        data = json.load(response)['data']
    endpoints = [e for e in data['endpoints'] if e.get('tag') == model.provider_route]
    if len(endpoints) != 1:
        raise RuntimeError(f'Pinned endpoint unavailable: {model.provider_route}')
    endpoint = endpoints[0]
    if endpoint.get('provider_name') != model.provider:
        raise RuntimeError('Endpoint provider identity changed')
    if model.quantization and endpoint.get('quantization') != model.quantization:
        raise RuntimeError('Endpoint quantization changed')
    needed = {'seed', 'temperature', 'max_tokens', 'response_format'}
    if model.reasoning_effort or model.reasoning_max_tokens:
        needed.add('reasoning')
    if not needed.issubset(endpoint.get('supported_parameters', [])):
        raise RuntimeError('Pinned endpoint no longer advertises the required parameters')
    signature = {k: endpoint.get(k) for k in
                 ('tag', 'provider_name', 'quantization', 'context_length', 'max_completion_tokens')}
    signature['supported_parameters'] = sorted(needed)
    signature['model'] = model.model
    anchor = OUT / (config['name'] + '.endpoint.json')
    with (OUT / (config['name'] + '.endpoint_history.jsonl')).open('a') as log:
        log.write(json.dumps({'time_utc': datetime.now(UTC).isoformat(), 'endpoint': endpoint}) + '\n')
    previous = json.loads(anchor.read_text()) if anchor.exists() else signature
    # Older anchors listed all advertised parameters; compare only those our requests use.
    previous['supported_parameters'] = sorted(needed.intersection(previous.get('supported_parameters', [])))
    if previous != signature:
        raise RuntimeError('Endpoint metadata changed during collection; retain logs and review before resuming')
    anchor.write_text(json.dumps(signature, indent=2) + '\n')



In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

from sprint import analyze


# Budgets count ALL attempts already logged, so reruns do not renew them.
def run_family(family):
    canary = f'configs/canary_{family}.toml'
    pilot = f'configs/main_pilot_{family}.toml'
    endpoint_snapshot(load_config(pilot))
    console = OUT / f'{family}.console.log'
    print(f'{family}: starting; progress and errors -> {console}', flush=True)
    with console.open('a', buffering=1) as log:
        def command(module, config, budget, *flags):
            sh([sys.executable, '-u', '-m', module, config, '--out', str(OUT),
                '--budget', str(budget), *flags], stdout=log, stderr=subprocess.STDOUT)
        command('sprint.run', canary, 100)
        command('sprint.run', pilot, 10, '--context-probe')
        checks = analyze.launch_checks(OUT / f'canary_{family}', OUT / f'main_{family}_probe')
        print(f'{family}: {checks}', flush=True)
        if not all(checks.values()):
            raise RuntimeError(f'{family}: launch checks failed; see {console}')
        command('sprint.competence', pilot, 60)
        diagnostics = json.loads((OUT / f'main_{family}_competence' / 'summary.json').read_text())
        correct = sum(r.get('correct', False) for r in diagnostics['arithmetic'])
        print(f'{family}: arithmetic {correct}/20 correct; pilot starting. This is descriptive, not an exclusion gate.', flush=True)
        command('sprint.run', pilot, 3500)
        endpoint_snapshot(load_config(pilot))
    print(f'{family}: finished', flush=True)
    return family

failures = {}
with ThreadPoolExecutor(max_workers=2) as pool:
    jobs = {pool.submit(run_family, family): family for family in FAMILIES}
    for future in as_completed(jobs):
        family = jobs[future]
        try:
            future.result()
        except Exception as error:  # noqa: BLE001 -- report each failed family, then raise after exporting
            failures[family] = str(error)
            print(f'{family}: STOPPED: {error}', flush=True)
print('Family failures:', failures or 'none')
# Incomplete games remain on Drive and in the tables below. Do not discard or replace them.


In [ ]:
import pandas as pd

run_dirs = [OUT / f'main_{family}' for family in FAMILIES]
table = analyze.games(run_dirs)
if not table.empty:
    display(table)
    display(analyze.summary(table))
    table.to_csv(OUT / 'pilot_games.csv', index=False)
    analyze.summary(table).to_csv(OUT / 'pilot_summary.csv', index=False)
for family, run_dir in zip(FAMILIES, run_dirs):
    diagnostic_dir = OUT / f'main_{family}_competence'
    for name in ('arithmetic.csv', 'pricing.csv'):
        path = diagnostic_dir / name
        if path.exists():
            print(family, name)
            display(pd.read_csv(path))
    saved = diagnostic_dir / 'summary.json'
    if saved.exists():
        summary = analyze.competence_summary(json.loads(saved.read_text()))
        for label, rows in (('arithmetic_summary', [summary['arithmetic']]),
                            ('pricing_summary', summary['pricing'])):
            frame = pd.DataFrame(rows)
            print(family, label)
            display(frame)
            frame.to_csv(diagnostic_dir / (label + '.csv'), index=False)
    gaps = [analyze.best_response_gaps(p) for p in sorted(run_dir.glob('game_*.jsonl'))
            if not p.name.endswith('.calls.jsonl')]
    if gaps:
        pd.concat(gaps, ignore_index=True).to_csv(run_dir / 'best_response_gaps.csv', index=False)
print('No expansion launched. Review eligibility, costs, errors and runtime before sizing further collection.')
if failures:
    raise RuntimeError('Some families stopped; inspect their console logs. Existing games remain saved.')
